# 00 — 통합 데이터 전처리와 T1/T2 동결

> **선택 실행용 실험 기반 코드입니다.** 최신 최종 실험 명세 전체를 구현한 상태는 아닙니다. 실제 본 실험 전 [`docs/ml/README.md`의 구현 차이](../../../docs/ml/README.md#implementation-gaps)를 확인하고 필요한 보완을 완료하세요. 이 노트북의 설정을 최종 연구 기준으로 사용하지 않습니다.

RouterBench와 xRouteBench를 정제·중복 제거하고, **train-only 5-fold group OOF**로 9개 threshold 후보 중 T1/T2를 고릅니다. RouterArena 라벨은 선택에 사용하지 않습니다.

완료 조건은 Drive의 `dataset/manifest.json`과 `dataset/_SUCCESS`입니다. coverage `resolution.json`이 `requires_resolution`이면 GPU 학습을 시작하지 마세요.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
archive = '/content/drive/MyDrive/LLM_Router/colab-inputs/router-model-comparison-colab-code.tar.gz'
print('코드 묶음:', archive)

In [ ]:
!rm -rf /content/LLM_Router
!tar -xzf "{archive}" -C /content
%cd /content/LLM_Router
!pip -q install -r ml/requirements-comparison.lock
!pip -q install --no-deps -e ml
import json, os, pathlib, shutil, subprocess, time
print('환경 준비 완료')

In [ ]:
# 원본은 immutable revision/size/SHA-256을 모두 검증합니다.
!routerbench-labeling --config ml/config/routerbench_labeling_v3.json download --data-dir ml/data/raw
!routerbench-labeling --config ml/config/routerbench_labeling_v3.json label --data-dir ml/data/raw --trust-pickle
!routerbench-labeling --config ml/config/routerbench_labeling_v3.json preprocess --profile full --seed 42
!routing-comparison-data --config ml/config/router_model_comparison_v1.json download-xroutebench
!routerarena-preprocessing download --data-dir ml/data/raw/routerarena --split full
!routerarena-preprocessing preprocess --data-dir ml/data/raw/routerarena --output-dir ml/artifacts/routerarena-evaluation-full --split full --seed 42 --training-path ml/artifacts/routerbench-training/withmartian-routerbench-2024-03-27__overall-solve-rate-v3-probe-selected__routerbench-training-preprocess-v3-natural/training.parquet

In [ ]:
# CPU 2~5시간 예상. 런타임이 끊기면 이 셀부터 다시 실행할 수 있습니다.
LOCAL_DATASET = pathlib.Path('/content/router-model-comparison-v1/dataset')
DRIVE_ROOT = pathlib.Path('/content/drive/MyDrive/LLM_Router/router-model-comparison/router-model-comparison-v1')
!routing-comparison-data --config ml/config/router_model_comparison_v1.json build --project-root /content/LLM_Router --xroute-data-dir ml/data/raw/xroutebench/ea4b6e1b29d9a734f55f0a637baf326bad6aa681 --routerarena-path ml/artifacts/routerarena-evaluation-full/evaluation.parquet --output-dir {LOCAL_DATASET}
selection = json.loads((LOCAL_DATASET / 'threshold/selected-t1-t2.json').read_text())
resolution = json.loads((LOCAL_DATASET / 'coverage/resolution.json').read_text())
print(json.dumps(selection, indent=2, ensure_ascii=False))
print('coverage:', resolution['status'], 'unresolved:', len(resolution['unresolved']))

In [ ]:
# 모든 checksum을 만든 뒤 staging -> final rename으로 Drive에 게시합니다.
if resolution['status'] != 'complete':
    raise RuntimeError('coverage resolution이 끝나지 않아 GPU 학습 입력을 봉인하지 않습니다.')
staging = DRIVE_ROOT / 'dataset.staging'
final = DRIVE_ROOT / 'dataset'
if staging.exists():
    shutil.rmtree(staging)
staging.parent.mkdir(parents=True, exist_ok=True)
shutil.copytree(LOCAL_DATASET, staging)
if final.exists():
    existing = json.loads((final / 'manifest.json').read_text())
    current = json.loads((staging / 'manifest.json').read_text())
    if existing['artifacts'] != current['artifacts']:
        raise RuntimeError('같은 experiment-id에 다른 dataset이 이미 있습니다.')
    shutil.rmtree(staging)
else:
    os.replace(staging, final)
(final / '_SUCCESS').write_text('ok\n')
print('봉인 완료:', final)

## 다음 실행

`01_tfidf_logreg.ipynb`에서 CPU 기준선을 실행할 수 있습니다. 그 이후 표현/모델 비교와 봉인 평가 노트북은 아직 구현되지 않았습니다. 후속 구현과 실행 조건은 [ML 최종 실험 명세](../../../docs/ml/README.md#implementation-gaps)를 따릅니다.
